In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/name-that-llm/sample_submission.parquet
/kaggle/input/competitions/name-that-llm/train.csv
/kaggle/input/competitions/name-that-llm/test.csv


In [ ]:
# ==========================================
# 1. Environment and Package Setup
# ==========================================
import os
import gc
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.preprocessing import LabelEncoder

gc.collect()
torch.cuda.empty_cache()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# ==========================================
# 2. Load Kaggle Datasets
# ==========================================
train_df = pd.read_csv('/kaggle/input/competitions/name-that-llm/train.csv')
print(f"Train set: {train_df.shape}")

# ==========================================
# 3. Label Encoding
# ==========================================
le = LabelEncoder()
train_df['label'] = le.fit_transform(train_df['model'])
num_labels = len(le.classes_)
print(f"Number of target model classes: {num_labels}")

# Save class mapping for Notebook 2
np.save('/kaggle/working/classes.npy', le.classes_)

# ==========================================
# 4. Tokenization & Dataset Split
# ==========================================
MODEL_NAME = 'roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

shuffled_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
train_split = shuffled_df.sample(frac=0.85, random_state=42)
val_split = shuffled_df.drop(train_split.index)

train_dataset = Dataset.from_pandas(train_split[['text', 'label']])
val_dataset = Dataset.from_pandas(val_split[['text', 'label']])

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

# ==========================================
# 5. Metrics & Model Definition
# ==========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = np.mean(predictions == labels)
    return {"accuracy": accuracy}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)

# ==========================================
# 6. Model Training (Fine-tuning)
# ==========================================
training_args = TrainingArguments(
    output_dir='/kaggle/working/results',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    warmup_steps=100,
    max_grad_norm=1.0,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    fp16=torch.cuda.is_available(),
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Starting model training run...")
trainer.train()

# ==========================================
# 7. Save Fine-Tuned Model & Tokenizer
# ==========================================
FINAL_DIR = '/kaggle/working/roberta_llm_classifier'
os.makedirs(FINAL_DIR, exist_ok=True)

# Save the trained model checkpoint and vocabulary
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print(f"Artifacts successfully saved to {FINAL_DIR}:")
print(os.listdir(FINAL_DIR))
print(f"Class mapping saved to /kaggle/working/classes.npy")

Using device: cpu
Train set: (2699, 4)
Number of target model classes: 18


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting model training run...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
